In [9]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

PROJECT_PATH = Path("..")
DATABASE_PATH = PROJECT_PATH / "data" / "fraud_analysis.db"
SQL_PATH = PROJECT_PATH / "sql"

engine = create_engine(f"sqlite:///{DATABASE_PATH}")

print("SQL Analysis Layer")
print("=" * 40)

print("Database:")
print(DATABASE_PATH.resolve())

print("\nSQL directory:")
print(SQL_PATH.resolve())

with engine.connect() as connection:
    sqlite_version = connection.execute(
        text("SELECT sqlite_version()")
    ).scalar()

print("\nSQLite version:", sqlite_version)
print("Connection successful.")

SQL Analysis Layer
Database:
D:\fraud-loss-analysis-step1\fraud-loss-analysis\data\fraud_analysis.db

SQL directory:
D:\fraud-loss-analysis-step1\fraud-loss-analysis\sql

SQLite version: 3.45.1
Connection successful.


In [10]:
# Helper function to read and execute SQL files

def run_sql_file(filename):
    """
    Read a SQL file from the sql/ directory
    and execute it against the SQLite database.
    """
    
    sql_file = SQL_PATH / filename

    if not sql_file.exists():
        raise FileNotFoundError(
            f"SQL file not found: {sql_file}"
        )

    sql_query = sql_file.read_text(encoding="utf-8")

    print(f"Executing: {filename}")
    print("-" * 50)

    with engine.connect() as connection:
        result = connection.execute(text(sql_query))
        rows = result.fetchall()
        columns = result.keys()

    return pd.DataFrame(rows, columns=columns)


print("SQL execution helper created successfully.")

SQL execution helper created successfully.


In [11]:
baseline_sql_file = SQL_PATH / "01_baseline_analysis.sql"

print("SQL file exists:", baseline_sql_file.exists())
print("SQL file:", baseline_sql_file.resolve())

sql_text = baseline_sql_file.read_text(encoding="utf-8")

print("\nNumber of SELECT statements:")
print(sql_text.upper().count("SELECT"))

print("\nSQL file preview:")
print(sql_text[:1000])

SQL file exists: True
SQL file: D:\fraud-loss-analysis-step1\fraud-loss-analysis\sql\01_baseline_analysis.sql

Number of SELECT statements:
3

SQL file preview:
-- ============================================================
-- 01_baseline_analysis.sql
-- Credit Card Fraud Pattern and Loss Analysis
-- ============================================================


-- ============================================================
-- QUERY 1: Overall Transaction & Fraud Baseline
-- ============================================================

SELECT
    COUNT(*) AS total_transactions,

    ROUND(SUM(amount), 2) AS total_transaction_value,

    ROUND(AVG(amount), 2) AS average_transaction_value,

    ROUND(
        SUM(is_fraud),
        0
    ) AS fraud_transactions,

    ROUND(
        SUM(
            CASE
                WHEN is_fraud = 1 THEN amount
                ELSE 0
            END
        ),
        2
    ) AS fraud_value,

    ROUND(
        100.0 * SUM(is_fraud) / COUNT(*),
   

In [12]:
# Read the baseline SQL file
baseline_sql = baseline_sql_file.read_text(encoding="utf-8")

# Split the file into individual SQL statements
baseline_queries = [
    query.strip()
    for query in baseline_sql.split(";")
    if query.strip()
]

print("Number of SQL queries found:", len(baseline_queries))

# Execute each query separately
baseline_results = []

for i, query in enumerate(baseline_queries, start=1):

    with engine.connect() as connection:
        result = connection.execute(text(query))

        rows = result.fetchall()
        columns = result.keys()

    df_result = pd.DataFrame(rows, columns=columns)

    baseline_results.append(df_result)

    print(f"\nQuery {i} result:")
    display(df_result)

Number of SQL queries found: 3

Query 1 result:


,total_transactions,total_transaction_value,average_transaction_value,fraud_transactions,fraud_value,fraud_rate_pct,fraud_value_pct
0,1498353,3.762811e+09,2511.3,10488.0,95197860.34,0.7,2.53



Query 2 result:


,total_chargebacks,gross_chargeback_loss,recovered_amount,net_chargeback_loss,recovery_rate_pct
0,10500,95285937.42,26506448.86,68779488.56,27.82



Query 3 result:


,year,month,month_name,total_transactions,fraud_transactions,transaction_value,fraud_value,fraud_rate_pct
0,2025,1,January,127340,926,3.080379e+08,7783767.62,0.73
1,2025,2,February,115381,773,2.795845e+08,6313373.34,0.67
2,2025,3,March,127437,906,3.127468e+08,7868828.94,0.71
3,2025,4,April,123235,850,2.982542e+08,7582866.01,0.69
4,2025,5,May,127197,897,3.097503e+08,7935379.90,0.71
5,2025,6,June,123669,884,3.023861e+08,8747508.26,0.71
6,2025,7,July,127143,873,3.104126e+08,9106609.84,0.69
7,2025,8,August,126537,855,3.080206e+08,8070341.27,0.68
8,2025,9,September,122745,859,2.975534e+08,7781842.53,0.70
9,2025,10,October,127325,897,3.675615e+08,7614749.13,0.70


In [13]:
# Query 1: Fraud concentration by channel

channel_sql = """
SELECT
    channel,
    COUNT(*) AS transactions,
    ROUND(SUM(amount), 2) AS transaction_value,
    SUM(is_fraud) AS fraud_transactions,

    ROUND(
        SUM(
            CASE
                WHEN is_fraud = 1
                THEN amount
                ELSE 0
            END
        ),
        2
    ) AS fraud_value,

    ROUND(
        100.0 * SUM(is_fraud) / COUNT(*),
        2
    ) AS fraud_rate_pct,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN is_fraud = 1
                THEN amount
                ELSE 0
            END
        )
        / SUM(amount),
        2
    ) AS fraud_value_pct

FROM fact_transaction

GROUP BY channel

ORDER BY fraud_value DESC;
"""

with engine.connect() as connection:
    channel_analysis = pd.read_sql(
        text(channel_sql),
        connection
    )

display(channel_analysis)

,channel,transactions,transaction_value,fraud_transactions,fraud_value,fraud_rate_pct,fraud_value_pct
0,E_COMMERCE,572928,1.475483e+09,7705,79988283.24,1.34,5.42
1,POS,626875,1.547832e+09,1893,10304227.08,0.30,0.67
2,CONTACTLESS,178820,4.417815e+08,515,2828225.78,0.29,0.64
3,ATM,119730,2.977145e+08,375,2077124.24,0.31,0.70


In [14]:
# Query 2: Fraud concentration by category

category_sql = """
SELECT
    category,
    COUNT(*) AS transactions,
    ROUND(SUM(amount), 2) AS transaction_value,
    SUM(is_fraud) AS fraud_transactions,

    ROUND(
        SUM(
            CASE
                WHEN is_fraud = 1
                THEN amount
                ELSE 0
            END
        ),
        2
    ) AS fraud_value,

    ROUND(
        100.0 * SUM(is_fraud) / COUNT(*),
        2
    ) AS fraud_rate_pct,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN is_fraud = 1
                THEN amount
                ELSE 0
            END
        )
        / SUM(amount),
        2
    ) AS fraud_value_pct

FROM fact_transaction

GROUP BY category

ORDER BY fraud_value DESC;
"""

with engine.connect() as connection:
    category_analysis = pd.read_sql(
        text(category_sql),
        connection
    )

display(category_analysis)

,category,transactions,transaction_value,fraud_transactions,fraud_value,fraud_rate_pct,fraud_value_pct
0,Electronics,125175,3.407701e+08,1820,37195582.66,1.45,10.92
1,Gift Cards,137426,3.719204e+08,1863,34719369.94,1.36,9.34
2,Travel,161689,3.975493e+08,895,3010383.57,0.55,0.76
3,Clothing,130096,3.225372e+08,733,2720082.57,0.56,0.84
4,Healthcare,135453,3.335084e+08,748,2653326.91,0.55,0.80
5,Grocery,137227,3.385309e+08,732,2628546.93,0.53,0.78
6,Entertainment,155294,3.824165e+08,827,2626975.45,0.53,0.69
7,Fuel,135610,3.353241e+08,753,2498938.23,0.56,0.75
8,Online Services,129502,3.192752e+08,728,2482512.76,0.56,0.78
9,Dining,127091,3.152100e+08,675,2403123.78,0.53,0.76


In [15]:
# Query 3: Fraud concentration by transaction amount

amount_sql = """
SELECT
    CASE
        WHEN amount < 500 THEN '0-500'
        WHEN amount < 1000 THEN '500-1K'
        WHEN amount < 5000 THEN '1K-5K'
        WHEN amount < 10000 THEN '5K-10K'
        WHEN amount < 25000 THEN '10K-25K'
        WHEN amount < 50000 THEN '25K-50K'
        WHEN amount < 100000 THEN '50K-100K'
        ELSE '100K+'
    END AS amount_band,

    COUNT(*) AS transactions,
    SUM(is_fraud) AS fraud_transactions,

    ROUND(SUM(amount), 2) AS transaction_value,

    ROUND(
        SUM(
            CASE
                WHEN is_fraud = 1
                THEN amount
                ELSE 0
            END
        ),
        2
    ) AS fraud_value,

    ROUND(
        100.0 * SUM(is_fraud) / COUNT(*),
        2
    ) AS fraud_rate_pct,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN is_fraud = 1
                THEN amount
                ELSE 0
            END
        )
        /
        (
            SELECT SUM(amount)
            FROM fact_transaction
            WHERE is_fraud = 1
        ),
        2
    ) AS fraud_value_share_pct

FROM fact_transaction

GROUP BY amount_band

ORDER BY
    CASE amount_band
        WHEN '0-500' THEN 1
        WHEN '500-1K' THEN 2
        WHEN '1K-5K' THEN 3
        WHEN '5K-10K' THEN 4
        WHEN '10K-25K' THEN 5
        WHEN '25K-50K' THEN 6
        WHEN '50K-100K' THEN 7
        WHEN '100K+' THEN 8
    END;
"""

with engine.connect() as connection:
    amount_analysis = pd.read_sql(
        text(amount_sql),
        connection
    )

display(amount_analysis)

,amount_band,transactions,fraud_transactions,transaction_value,fraud_value,fraud_rate_pct,fraud_value_share_pct
0,0-500,207966,2480,6.704072e+07,201643.61,1.19,0.21
1,500-1K,308875,813,2.292324e+08,610256.95,0.26,0.64
2,1K-5K,808186,3243,1.843204e+09,8208330.89,0.40,8.62
3,5K-10K,127636,1324,8.667389e+08,9285564.52,1.04,9.75
4,10K-25K,40896,1483,5.760152e+08,23708286.11,3.63,24.90
5,25K-50K,4147,813,1.349828e+08,28357680.13,19.60,29.79
6,50K-100K,584,284,3.740354e+07,18431871.52,48.63,19.36
7,100K+,63,48,8.193302e+06,6394226.61,76.19,6.72


In [16]:
# ============================================================
# QUERY 4: Channel × Category Fraud Concentration
# ============================================================

channel_category_sql = """
SELECT
    channel,
    category,

    COUNT(*) AS transactions,

    ROUND(
        SUM(amount),
        2
    ) AS transaction_value,

    SUM(is_fraud) AS fraud_transactions,

    ROUND(
        SUM(
            CASE
                WHEN is_fraud = 1 THEN amount
                ELSE 0
            END
        ),
        2
    ) AS fraud_value,

    ROUND(
        100.0 * SUM(is_fraud) / COUNT(*),
        2
    ) AS fraud_rate_pct,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN is_fraud = 1 THEN amount
                ELSE 0
            END
        )
        /
        (
            SELECT SUM(amount)
            FROM fact_transaction
            WHERE is_fraud = 1
        ),
        2
    ) AS fraud_value_share_pct

FROM fact_transaction

GROUP BY
    channel,
    category

ORDER BY
    fraud_value DESC;
"""

channel_category_analysis = pd.read_sql(
    channel_category_sql,
    engine
)

display(channel_category_analysis)

,channel,category,transactions,transaction_value,fraud_transactions,fraud_value,fraud_rate_pct,fraud_value_share_pct
0,E_COMMERCE,Electronics,48494,1.511111e+08,1594,35887039.34,3.29,37.70
1,E_COMMERCE,Gift Cards,53422,1.627231e+08,1612,33249258.30,3.02,34.93
2,E_COMMERCE,Online Services,49422,1.220305e+08,500,1274967.89,1.01,1.34
3,E_COMMERCE,Travel,61667,1.513200e+08,575,1274021.89,0.93,1.34
4,E_COMMERCE,Grocery,52213,1.286901e+08,459,1238648.70,0.88,1.30
5,E_COMMERCE,Entertainment,59101,1.449017e+08,576,1229881.75,0.97,1.29
6,E_COMMERCE,Clothing,49704,1.229515e+08,474,1223943.43,0.95,1.29
7,E_COMMERCE,Healthcare,51438,1.266873e+08,491,1197916.82,0.95,1.26
8,E_COMMERCE,Fuel,51895,1.278920e+08,507,1197281.69,0.98,1.26
9,E_COMMERCE,Dining,48211,1.195381e+08,435,1181999.96,0.90,1.24


In [17]:
# ============================================================
# QUERY 5: High-Value E-Commerce Risk Pattern
# ============================================================

high_value_pattern_sql = """
SELECT
    CASE
        WHEN amount >= 10000 THEN '10K+'
        ELSE '<10K'
    END AS amount_segment,

    COUNT(*) AS transactions,

    SUM(is_fraud) AS fraud_transactions,

    ROUND(
        SUM(amount),
        2
    ) AS transaction_value,

    ROUND(
        SUM(
            CASE
                WHEN is_fraud = 1 THEN amount
                ELSE 0
            END
        ),
        2
    ) AS fraud_value,

    ROUND(
        100.0 * SUM(is_fraud) / COUNT(*),
        2
    ) AS fraud_rate_pct,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN is_fraud = 1 THEN amount
                ELSE 0
            END
        )
        /
        (
            SELECT SUM(amount)
            FROM fact_transaction
            WHERE is_fraud = 1
        ),
        2
    ) AS fraud_value_share_pct

FROM fact_transaction

WHERE
    channel = 'E_COMMERCE'
    AND category IN ('Electronics', 'Gift Cards')

GROUP BY
    amount_segment

ORDER BY
    CASE
        WHEN amount_segment = '10K+' THEN 1
        ELSE 2
    END;
"""

high_value_pattern = pd.read_sql(
    high_value_pattern_sql,
    engine
)

display(high_value_pattern)

,amount_segment,transactions,fraud_transactions,transaction_value,fraud_value,fraud_rate_pct,fraud_value_share_pct
0,10K+,4938,1987,1.123212e+08,65862706.81,40.24,69.19
1,<10K,96978,1219,2.015130e+08,3273590.83,1.26,3.44


In [18]:
# ============================================================
# QUERY 6: Fraud Rule Threshold Sweep
# ============================================================

threshold_sweep_sql = """
WITH thresholds AS (

    SELECT 10000 AS threshold
    UNION ALL
    SELECT 12500
    UNION ALL
    SELECT 15000
    UNION ALL
    SELECT 17500
    UNION ALL
    SELECT 20000
    UNION ALL
    SELECT 22500
    UNION ALL
    SELECT 25000
    UNION ALL
    SELECT 30000
    UNION ALL
    SELECT 40000
    UNION ALL
    SELECT 50000

)

SELECT
    t.threshold,

    COUNT(f.txn_id) AS alert_volume,

    SUM(f.is_fraud) AS fraud_transactions,

    COUNT(f.txn_id) - SUM(f.is_fraud) AS legitimate_transactions,

    ROUND(
        100.0 * SUM(f.is_fraud) / COUNT(f.txn_id),
        2
    ) AS precision_pct,

    ROUND(
        100.0 *
        (COUNT(f.txn_id) - SUM(f.is_fraud))
        / COUNT(f.txn_id),
        2
    ) AS false_positive_pct,

    ROUND(
        SUM(
            CASE
                WHEN f.is_fraud = 1 THEN f.amount
                ELSE 0
            END
        ),
        2
    ) AS fraud_value_caught,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN f.is_fraud = 1 THEN f.amount
                ELSE 0
            END
        )
        /
        (
            SELECT SUM(amount)
            FROM fact_transaction
            WHERE is_fraud = 1
        ),
        2
    ) AS fraud_value_coverage_pct

FROM thresholds t

LEFT JOIN fact_transaction f
    ON f.channel = 'E_COMMERCE'
    AND f.category IN ('Electronics', 'Gift Cards')
    AND f.amount >= t.threshold

GROUP BY
    t.threshold

ORDER BY
    t.threshold;
"""

threshold_sweep = pd.read_sql(
    threshold_sweep_sql,
    engine
)

display(threshold_sweep)

,threshold,alert_volume,fraud_transactions,legitimate_transactions,precision_pct,false_positive_pct,fraud_value_caught,fraud_value_coverage_pct
0,10000,4938,1987,2951,40.24,59.76,65862706.81,69.19
1,12500,3528,1807,1721,51.22,48.78,63853800.22,67.07
2,15000,2691,1627,1064,60.46,39.54,61391754.17,64.49
3,17500,2177,1464,713,67.25,32.75,58750068.30,61.71
4,20000,1793,1318,475,73.51,26.49,56011958.64,58.84
5,22500,1516,1183,333,78.03,21.97,53143559.23,55.82
6,25000,1314,1065,249,81.05,18.95,50330951.71,52.87
7,30000,978,840,138,85.89,14.11,44157536.69,46.39
8,40000,585,528,57,90.26,9.74,33406649.89,35.09
9,50000,344,322,22,93.60,6.40,24201410.45,25.42


In [19]:
# ============================================================
# QUERY 7: Development vs Validation
# ============================================================

development_validation_sql = """
WITH thresholds AS (

    SELECT 10000 AS threshold
    UNION ALL
    SELECT 12500
    UNION ALL
    SELECT 15000
    UNION ALL
    SELECT 17500
    UNION ALL
    SELECT 20000
    UNION ALL
    SELECT 22500
    UNION ALL
    SELECT 25000
    UNION ALL
    SELECT 30000
    UNION ALL
    SELECT 40000
    UNION ALL
    SELECT 50000

),

base AS (

    SELECT
        txn_id,
        amount,
        is_fraud,

        CASE
            WHEN strftime('%m', txn_ts) BETWEEN '01' AND '08'
                THEN 'DEVELOPMENT'
            ELSE 'VALIDATION'
        END AS dataset_split

    FROM fact_transaction

    WHERE
        channel = 'E_COMMERCE'
        AND category IN ('Electronics', 'Gift Cards')
)

SELECT
    t.threshold,
    b.dataset_split,

    COUNT(b.txn_id) AS alert_volume,

    SUM(b.is_fraud) AS fraud_transactions,

    COUNT(b.txn_id) - SUM(b.is_fraud)
        AS legitimate_transactions,

    ROUND(
        100.0 * SUM(b.is_fraud) / COUNT(b.txn_id),
        2
    ) AS precision_pct,

    ROUND(
        SUM(
            CASE
                WHEN b.is_fraud = 1 THEN b.amount
                ELSE 0
            END
        ),
        2
    ) AS fraud_value_caught

FROM thresholds t

JOIN base b
    ON b.amount >= t.threshold

GROUP BY
    t.threshold,
    b.dataset_split

ORDER BY
    t.threshold,
    b.dataset_split;
"""

development_validation = pd.read_sql(
    development_validation_sql,
    engine
)

display(development_validation)

,threshold,dataset_split,alert_volume,fraud_transactions,legitimate_transactions,precision_pct,fraud_value_caught
0,10000,DEVELOPMENT,3135,1339,1796,42.71,44037512.82
1,10000,VALIDATION,1803,648,1155,35.94,21825193.99
2,12500,DEVELOPMENT,2255,1203,1052,53.35,42524559.16
3,12500,VALIDATION,1273,604,669,47.45,21329241.06
4,15000,DEVELOPMENT,1731,1084,647,62.62,40897148.51
5,15000,VALIDATION,960,543,417,56.56,20494605.66
6,17500,DEVELOPMENT,1412,976,436,69.12,39146280.26
7,17500,VALIDATION,765,488,277,63.79,19603788.04
8,20000,DEVELOPMENT,1177,886,291,75.28,37461756.66
9,20000,VALIDATION,616,432,184,70.13,18550201.98


In [20]:
# ============================================================
# BUSINESS ASSUMPTIONS
# ============================================================

business_assumptions = {
    "review_cost_per_alert": 150,
    "customer_friction_per_legit_alert": 100,
    "fraud_prevention_rate": 0.80,
    "review_capacity_per_month": 500
}

business_assumptions

{'review_cost_per_alert': 150,
 'customer_friction_per_legit_alert': 100,
 'fraud_prevention_rate': 0.8,
 'review_capacity_per_month': 500}

In [21]:
# ============================================================
# COST-BENEFIT ANALYSIS
# ============================================================

validation_results = development_validation[
    development_validation["dataset_split"] == "VALIDATION"
].copy()

# Monthly alert volume
validation_months = 4

validation_results["alerts_per_month"] = (
    validation_results["alert_volume"] / validation_months
)

# Expected fraud prevented
validation_results["expected_fraud_prevented_value"] = (
    validation_results["fraud_value_caught"]
    * business_assumptions["fraud_prevention_rate"]
)

# Review cost
validation_results["review_cost"] = (
    validation_results["alert_volume"]
    * business_assumptions["review_cost_per_alert"]
)

# Customer friction cost
validation_results["customer_friction_cost"] = (
    validation_results["legitimate_transactions"]
    * business_assumptions["customer_friction_per_legit_alert"]
)

# Expected net savings
validation_results["net_savings"] = (
    validation_results["expected_fraud_prevented_value"]
    - validation_results["review_cost"]
    - validation_results["customer_friction_cost"]
)

# Review capacity utilization
validation_results["capacity_utilization_pct"] = (
    validation_results["alerts_per_month"]
    / business_assumptions["review_capacity_per_month"]
    * 100
)

cost_benefit_columns = [
    "threshold",
    "alert_volume",
    "fraud_transactions",
    "legitimate_transactions",
    "precision_pct",
    "fraud_value_caught",
    "expected_fraud_prevented_value",
    "review_cost",
    "customer_friction_cost",
    "net_savings",
    "alerts_per_month",
    "capacity_utilization_pct"
]

display(
    validation_results[cost_benefit_columns]
)

,threshold,alert_volume,fraud_transactions,legitimate_transactions,precision_pct,fraud_value_caught,expected_fraud_prevented_value,review_cost,customer_friction_cost,net_savings,alerts_per_month,capacity_utilization_pct
1,10000,1803,648,1155,35.94,21825193.99,1.746016e+07,270450,115500,1.707421e+07,450.75,90.15
3,12500,1273,604,669,47.45,21329241.06,1.706339e+07,190950,66900,1.680554e+07,318.25,63.65
5,15000,960,543,417,56.56,20494605.66,1.639568e+07,144000,41700,1.620998e+07,240.00,48.00
7,17500,765,488,277,63.79,19603788.04,1.568303e+07,114750,27700,1.554058e+07,191.25,38.25
9,20000,616,432,184,70.13,18550201.98,1.484016e+07,92400,18400,1.472936e+07,154.00,30.80
11,22500,513,390,123,76.02,17657343.52,1.412587e+07,76950,12300,1.403662e+07,128.25,25.65
13,25000,446,349,97,78.25,16676431.02,1.334114e+07,66900,9700,1.326454e+07,111.50,22.30
15,30000,327,276,51,84.40,14665987.27,1.173279e+07,49050,5100,1.167864e+07,81.75,16.35
17,40000,199,174,25,87.44,11171381.65,8.937105e+06,29850,2500,8.904755e+06,49.75,9.95
19,50000,114,105,9,92.11,8080463.11,6.464370e+06,17100,900,6.446370e+06,28.50,5.70


In [22]:
# ============================================================
# BUSINESS GUARDRAILS
# ============================================================

MIN_PRECISION = 70.0
MAX_CAPACITY_UTILIZATION = 50.0

eligible_rules = validation_results[
    (validation_results["precision_pct"] >= MIN_PRECISION)
    &
    (validation_results["capacity_utilization_pct"] <= MAX_CAPACITY_UTILIZATION)
].copy()

eligible_rules = eligible_rules.sort_values(
    "net_savings",
    ascending=False
)

display(
    eligible_rules[cost_benefit_columns]
)

,threshold,alert_volume,fraud_transactions,legitimate_transactions,precision_pct,fraud_value_caught,expected_fraud_prevented_value,review_cost,customer_friction_cost,net_savings,alerts_per_month,capacity_utilization_pct
9,20000,616,432,184,70.13,18550201.98,1.484016e+07,92400,18400,1.472936e+07,154.00,30.80
11,22500,513,390,123,76.02,17657343.52,1.412587e+07,76950,12300,1.403662e+07,128.25,25.65
13,25000,446,349,97,78.25,16676431.02,1.334114e+07,66900,9700,1.326454e+07,111.50,22.30
15,30000,327,276,51,84.40,14665987.27,1.173279e+07,49050,5100,1.167864e+07,81.75,16.35
17,40000,199,174,25,87.44,11171381.65,8.937105e+06,29850,2500,8.904755e+06,49.75,9.95
19,50000,114,105,9,92.11,8080463.11,6.464370e+06,17100,900,6.446370e+06,28.50,5.70


In [23]:
# ============================================================
# RECOMMENDED RULE
# ============================================================

recommended_rule = eligible_rules.iloc[0]

print("Recommended threshold:")
print(f"₹{recommended_rule['threshold']:,.0f}")

print("\nValidation precision:")
print(f"{recommended_rule['precision_pct']:.2f}%")

print("\nAlerts per month:")
print(f"{recommended_rule['alerts_per_month']:.2f}")

print("\nReview capacity utilization:")
print(f"{recommended_rule['capacity_utilization_pct']:.2f}%")

print("\nFraud value caught:")
print(f"₹{recommended_rule['fraud_value_caught']:,.2f}")

print("\nExpected fraud prevented:")
print(f"₹{recommended_rule['expected_fraud_prevented_value']:,.2f}")

print("\nExpected net savings:")
print(f"₹{recommended_rule['net_savings']:,.2f}")

Recommended threshold:
₹20,000

Validation precision:
70.13%

Alerts per month:
154.00

Review capacity utilization:
30.80%

Fraud value caught:
₹18,550,201.98

Expected fraud prevented:
₹14,840,161.58

Expected net savings:
₹14,729,361.58


In [24]:
# ============================================================
# SQL 03 - Velocity Analysis
# Execute each query separately
# ============================================================

velocity_sql = (SQL_PATH / "03_velocity_analysis.sql").read_text(
    encoding="utf-8"
)

# Split the file into individual SQL statements
velocity_queries = [
    query.strip()
    for query in velocity_sql.split(";")
    if query.strip()
]

print("Number of SQL queries found:", len(velocity_queries))

velocity_results = []

for i, query in enumerate(velocity_queries, start=1):

    with engine.connect() as connection:
        result = connection.execute(text(query))
        rows = result.fetchall()
        columns = result.keys()

    df_result = pd.DataFrame(rows, columns=columns)

    velocity_results.append(df_result)

    print(f"\n{'=' * 60}")
    print(f"QUERY {i}")
    print(f"{'=' * 60}")

    display(df_result)

Number of SQL queries found: 3

QUERY 1


,velocity_band,transactions,fraud_transactions,fraud_rate_pct,fraud_value
0,1 transaction,1496862,10482,0.7,95101485.26
1,2 transactions,1490,6,0.4,96375.08
2,3+ transactions,1,0,0.0,0.00



QUERY 2


,velocity_band,transactions,fraud_transactions,fraud_rate_pct,fraud_value
0,1 transaction,1489820,10429,0.70,94821109.19
1,2 transactions,8508,59,0.69,376751.15
2,3 transactions,25,0,0.00,0.00



QUERY 3


,velocity_band,transactions,fraud_transactions,fraud_rate_pct,fraud_value
0,1 transaction,1307312,9262,0.71,83713829.24
1,2 transactions,178444,1131,0.63,10713457.53
2,3 transactions,12036,91,0.76,762550.22
3,4-5 transactions,560,4,0.71,8023.35
4,6+ transactions,1,0,0.00,0.00


In [25]:
# ============================================================
# SQL 04 - Fraud Loss Analysis
# ============================================================

loss_sql = (SQL_PATH / "04_fraud_loss_analysis.sql").read_text(
    encoding="utf-8"
)

loss_queries = [
    query.strip()
    for query in loss_sql.split(";")
    if query.strip()
]

print("Number of SQL queries found:", len(loss_queries))

loss_results = []

for i, query in enumerate(loss_queries, start=1):

    with engine.connect() as connection:
        result = connection.execute(text(query))
        rows = result.fetchall()
        columns = result.keys()

    df_result = pd.DataFrame(rows, columns=columns)

    loss_results.append(df_result)

    print(f"\n{'=' * 60}")
    print(f"QUERY {i}")
    print(f"{'=' * 60}")

    display(df_result)

Number of SQL queries found: 6

QUERY 1


,confirmed_fraud_chargebacks,gross_chargeback_loss,recovered_amount,net_loss,recovery_rate_pct
0,10500,95285937.42,26506448.86,68779488.56,27.82



QUERY 2


,transaction_month,fraud_transactions,gross_fraud_value,recovered_amount,net_loss,recovery_vs_fraud_value_pct
0,2025-01,926,7783767.62,2161737.13,5622030.49,27.77
1,2025-02,773,6313373.34,1656416.45,4656956.89,26.24
2,2025-03,906,7868828.94,2134396.25,5734432.69,27.12
3,2025-04,850,7582866.01,2080875.62,5501990.39,27.44
4,2025-05,897,7935379.90,2209034.39,5726345.51,27.84
5,2025-06,884,8747508.26,2465311.01,6282197.25,28.18
6,2025-07,873,9106609.84,2664298.08,6442311.76,29.26
7,2025-08,855,8070341.27,2192311.37,5878029.90,27.17
8,2025-09,859,7781842.53,2118594.58,5663247.95,27.22
9,2025-10,897,7614749.13,2228923.10,5385826.03,29.27



QUERY 3


...

,category,fraud_transactions,gross_fraud_value,recovered_amount,net_loss,net_loss_share_pct
0,Electronics,1820,37195582.66,10508721.93,26686860.73,38.83
1,Gift Cards,1863,34719369.94,9624028.94,25095341.00,36.51
2,Travel,895,3010383.57,825682.77,2184700.80,3.18
3,Clothing,733,2720082.57,714476.14,2005606.43,2.92
4,Entertainment,827,2626975.45,698512.12,1928463.33,2.81
5,Healthcare,748,2653326.91,729700.99,1923625.92,2.80
6,Grocery,732,2628546.93,768163.37,1860383.56,2.71
7,Fuel,753,2498938.23,685359.16,1813579.07,2.64
8,Online Services,728,2482512.76,680700.77,1801811.99,2.62
9,Dining,675,2403123.78,638836.15,1764287.63,2.57



QUERY 5


,threshold,fraud_transactions,fraud_value,net_loss,net_loss_coverage_pct
0,10000,2628,76892064.37,55439969.71,80.67
1,20000,1461,60259247.61,43383515.89,63.12
2,25000,1145,53183778.26,38251805.38,55.66
3,50000,332,24826098.13,17607743.89,25.62



QUERY 6


,channel,category,fraud_transactions,gross_fraud_value,net_loss,total_net_loss_share_pct
0,E_COMMERCE,Electronics,1594,35887039.34,25769317.94,27.04
1,E_COMMERCE,Gift Cards,1612,33249258.30,24040936.53,25.23


In [26]:
# ============================================================
# SQL 05 - Rule Simulation
# ============================================================

rule_sql = (SQL_PATH / "05_rule_simulation.sql").read_text(
    encoding="utf-8"
)

rule_queries = [
    query.strip()
    for query in rule_sql.split(";")
    if query.strip()
]

print("Number of SQL queries found:", len(rule_queries))

rule_results = []

for i, query in enumerate(rule_queries, start=1):

    with engine.connect() as connection:
        result = connection.execute(text(query))
        rows = result.fetchall()
        columns = result.keys()

    df_result = pd.DataFrame(rows, columns=columns)

    rule_results.append(df_result)

    print(f"\n{'=' * 60}")
    print(f"QUERY {i}")
    print(f"{'=' * 60}")

    display(df_result)

Number of SQL queries found: 5

QUERY 1


,threshold,alert_volume,fraud_transactions,legitimate_transactions,precision_pct,fraud_value_caught,fraud_value_coverage_pct
0,10000,45690,2628,43062,5.75,76892064.37,80.77
1,12500,27515,2222,25293,8.08,72366594.59,76.02
2,15000,17672,1919,15753,10.86,68226938.52,71.67
3,17500,12142,1679,10463,13.83,64337450.34,67.58
4,20000,8591,1461,7130,17.01,60259247.61,63.30
5,22500,6334,1289,5045,20.35,56608193.30,59.46
6,25000,4794,1145,3649,23.88,53183778.26,55.87
7,30000,2883,886,1997,30.73,46077106.79,48.40
8,40000,1294,545,749,42.12,34337093.31,36.07
9,50000,647,332,315,51.31,24826098.13,26.08



QUERY 2


,channel,category_group,transactions,fraud_transactions,transaction_value,fraud_value,fraud_rate_pct
0,E_COMMERCE,High-risk category,101916,3206,3.138343e+08,69136297.64,3.15
1,E_COMMERCE,Other category,471012,4499,1.161648e+09,10851985.60,0.96
2,POS,Other category,517931,1557,1.276793e+09,8309324.85,0.30
3,CONTACTLESS,Other category,147843,437,3.661053e+08,2433922.07,0.30
4,POS,High-risk category,108944,336,2.710395e+08,1994902.23,0.31
5,ATM,Other category,98966,312,2.455739e+08,1687675.22,0.32
6,CONTACTLESS,High-risk category,30977,78,7.567624e+07,394303.71,0.25
7,ATM,High-risk category,20764,63,5.214052e+07,389449.02,0.30



QUERY 3


,threshold,alert_volume,fraud_transactions,legitimate_transactions,precision_pct,fraud_value_caught,pattern_fraud_value_coverage_pct
0,10000,4938,1987,2951,40.24,65862706.81,95.27
1,12500,3528,1807,1721,51.22,63853800.22,92.36
2,15000,2691,1627,1064,60.46,61391754.17,88.80
3,17500,2177,1464,713,67.25,58750068.30,84.98
4,20000,1793,1318,475,73.51,56011958.64,81.02
5,22500,1516,1183,333,78.03,53143559.23,76.87
6,25000,1314,1065,249,81.05,50330951.71,72.80
7,30000,978,840,138,85.89,44157536.69,63.87
8,40000,585,528,57,90.26,33406649.89,48.32
9,50000,344,322,22,93.60,24201410.45,35.01



QUERY 4


,alert_volume,fraud_transactions,legitimate_transactions,precision_pct,fraud_value_caught
0,1793,1318,475,73.51,56011958.64



QUERY 5


,transaction_month,alert_volume,fraud_transactions,fraud_value_caught,precision_pct
0,2025-01,142,101,4233433.87,71.13
1,2025-02,106,78,3441031.05,73.58
2,2025-03,157,125,4748642.80,79.62
3,2025-04,137,99,4260198.47,72.26
4,2025-05,143,111,4575343.64,77.62
5,2025-06,161,124,5482234.01,77.02
6,2025-07,180,139,6073518.75,77.22
7,2025-08,151,109,4647354.07,72.19
8,2025-09,148,112,4631929.83,75.68
9,2025-10,162,105,4293932.11,64.81


In [27]:
# ============================================================
# SQL 06 - Temporal & Cohort Analysis
# ============================================================

temporal_sql = (SQL_PATH / "06_temporal_cohort_analysis.sql").read_text(
    encoding="utf-8"
)

temporal_queries = [
    query.strip()
    for query in temporal_sql.split(";")
    if query.strip()
]

print("Number of SQL queries found:", len(temporal_queries))

temporal_results = []

for i, query in enumerate(temporal_queries, start=1):

    with engine.connect() as connection:
        result = connection.execute(text(query))
        rows = result.fetchall()
        columns = result.keys()

    df_result = pd.DataFrame(rows, columns=columns)

    temporal_results.append(df_result)

    print(f"\n{'=' * 60}")
    print(f"QUERY {i}")
    print(f"{'=' * 60}")

    display(df_result)

Number of SQL queries found: 5

QUERY 1


,transaction_month,transactions,transaction_value,fraud_transactions,fraud_value,fraud_rate_pct,fraud_value_share_pct
0,2025-01,127340,3.080379e+08,926,7783767.62,0.73,2.53
1,2025-02,115381,2.795845e+08,773,6313373.34,0.67,2.26
2,2025-03,127437,3.127468e+08,906,7868828.94,0.71,2.52
3,2025-04,123235,2.982542e+08,850,7582866.01,0.69,2.54
4,2025-05,127197,3.097503e+08,897,7935379.90,0.71,2.56
5,2025-06,123669,3.023861e+08,884,8747508.26,0.71,2.89
6,2025-07,127143,3.104126e+08,873,9106609.84,0.69,2.93
7,2025-08,126537,3.080206e+08,855,8070341.27,0.68,2.62
8,2025-09,122745,2.975534e+08,859,7781842.53,0.70,2.62
9,2025-10,127325,3.675615e+08,897,7614749.13,0.70,2.07



QUERY 2


,transaction_month,channel,transactions,fraud_transactions,fraud_value,fraud_rate_pct
0,2025-01,E_COMMERCE,48611,685,6489279.19,1.41
1,2025-01,POS,53447,164,997526.79,0.31
2,2025-01,CONTACTLESS,15188,44,150950.49,0.29
3,2025-01,ATM,10094,33,146011.15,0.33
4,2025-02,E_COMMERCE,44136,565,5252526.14,1.28
5,2025-02,POS,48096,146,769053.67,0.30
6,2025-02,ATM,9402,32,162855.67,0.34
7,2025-02,CONTACTLESS,13747,30,128937.86,0.22
8,2025-03,E_COMMERCE,48579,662,6646704.88,1.36
9,2025-03,POS,53582,172,771753.37,0.32



QUERY 3


,card_age_band,transactions,fraud_transactions,transaction_value,fraud_value,fraud_rate_pct
0,0-30 days,54185,372,1.364573e+08,3328712.00,0.69
1,31-90 days,99500,705,2.506366e+08,7569438.55,0.71
2,91-180 days,152708,1022,3.805806e+08,8571515.32,0.67
3,181-365 days,304760,2057,7.646237e+08,18210670.35,0.67
4,1-2 years,613187,4389,1.544761e+09,39757930.69,0.72
5,2+ years,274013,1943,6.857514e+08,17759593.43,0.71



QUERY 4


,card_age_band,channel,transactions,fraud_transactions,fraud_value,fraud_rate_pct
0,0-30 days,E_COMMERCE,20722,271,2823040.37,1.31
1,0-30 days,POS,22632,62,273829.42,0.27
2,0-30 days,CONTACTLESS,6500,27,142190.72,0.42
3,0-30 days,ATM,4331,12,89651.49,0.28
4,1-2 years,E_COMMERCE,233972,3260,33572038.10,1.39
5,1-2 years,POS,256702,776,4158087.82,0.30
6,1-2 years,CONTACTLESS,73176,200,1125414.72,0.27
7,1-2 years,ATM,49337,153,902390.05,0.31
8,181-365 days,E_COMMERCE,116645,1530,15235337.88,1.31
9,181-365 days,POS,127607,372,2115151.50,0.29



QUERY 5


,transaction_month,alert_volume,fraud_transactions,fraud_value_caught,precision_pct
0,2025-01,142,101,4233433.87,71.13
1,2025-02,106,78,3441031.05,73.58
2,2025-03,157,125,4748642.80,79.62
3,2025-04,137,99,4260198.47,72.26
4,2025-05,143,111,4575343.64,77.62
5,2025-06,161,124,5482234.01,77.02
6,2025-07,180,139,6073518.75,77.22
7,2025-08,151,109,4647354.07,72.19
8,2025-09,148,112,4631929.83,75.68
9,2025-10,162,105,4293932.11,64.81


In [28]:
sql_text = (SQL_PATH / "07_data_quality_checks.sql").read_text(encoding="utf-8")

queries = [q.strip() for q in sql_text.split(";") if q.strip()]

results = []

for i, query in enumerate(queries, start=1):
    with engine.connect() as connection:
        result = connection.execute(text(query))
        rows = result.fetchall()
        columns = result.keys()

    df_result = pd.DataFrame(rows, columns=columns)
    results.append(df_result)

    print(f"\n===== QUERY {i} =====")
    display(df_result)


===== QUERY 1 =====


,table_name,row_count
0,dim_customer,20000
1,dim_card,30000
2,dim_merchant,2000
3,dim_device,35001
4,fact_transaction,1498353
5,fact_chargeback,10500



===== QUERY 2 =====


,duplicate_txn_id_groups,duplicate_transaction_rows
0,0,0



===== QUERY 3 =====


...

,unknown_device_transactions
0,2994



===== QUERY 11 =====


,minimum_amount,average_amount,approximate_median_amount,maximum_amount
0,1.07,2511.3,1495.02,250000.0



===== QUERY 12 =====


,transaction_rows,orphan_customer_rows,orphan_card_rows,orphan_merchant_rows,orphan_device_rows,invalid_amount_rows,duplicate_txn_id_groups
0,1498353,0,0,0,0,0,0


In [35]:
sql_text = (SQL_PATH / "08_final_rule_results.sql").read_text(encoding="utf-8")

queries = [q.strip() for q in sql_text.split(";") if q.strip()]

results = []

for i, query in enumerate(queries, start=1):
    with engine.connect() as connection:
        result = connection.execute(text(query))
        rows = result.fetchall()
        columns = result.keys()

    df_result = pd.DataFrame(rows, columns=columns)
    results.append(df_result)

    print(f"\n===== QUERY {i} =====")
    display(df_result)


===== QUERY 1 =====


,rule_id,rule_name,alerts,fraud_caught,legitimate_alerts,precision_pct,fraud_value_caught
0,R001,E-commerce + high-risk category + amount >= 20K,1793,1318,475,73.51,56011958.64



===== QUERY 2 =====


,rule_id,rule_name,alerts,fraud_caught,legitimate_alerts,precision_pct,fraud_value_caught
0,R002,E-commerce card testing: amount <= ₹50,2230,2030,200,91.03,52580.32



===== QUERY 3 =====


,month,alerts,fraud_caught,legitimate_alerts,precision_pct,fraud_value_caught
0,2025-01,142,101,41,71.13,4233433.87
1,2025-02,106,78,28,73.58,3441031.05
2,2025-03,157,125,32,79.62,4748642.80
3,2025-04,137,99,38,72.26,4260198.47
4,2025-05,143,111,32,77.62,4575343.64
5,2025-06,161,124,37,77.02,5482234.01
6,2025-07,180,139,41,77.22,6073518.75
7,2025-08,151,109,42,72.19,4647354.07
8,2025-09,148,112,36,75.68,4631929.83
9,2025-10,162,105,57,64.81,4293932.11



===== QUERY 4 =====


,month,alerts,fraud_caught,legitimate_alerts,precision_pct,fraud_value_caught
0,2025-01,202,185,17,91.58,4795.75
1,2025-02,171,152,19,88.89,3936.34
2,2025-03,195,172,23,88.21,4421.97
3,2025-04,172,159,13,92.44,4143.58
4,2025-05,173,157,16,90.75,4200.34
5,2025-06,170,156,14,91.76,4024.57
6,2025-07,170,159,11,93.53,4172.95
7,2025-08,196,174,22,88.78,4390.97
8,2025-09,206,180,26,87.38,4599.99
9,2025-10,185,177,8,95.68,4337.46



===== QUERY 5 =====


,rule_id,rule_name,alerts,fraud_caught,legitimate_alerts,precision_pct,fraud_value_caught
0,R001,High-risk e-commerce >= ₹20K,1793,1318,475,73.51,56011958.64
1,R002,Card testing: e-commerce <= ₹50,2230,2030,200,91.03,52580.32
